In [ ]:
# The imports

from dotenv import load_dotenv
from agents.mcp import MCPServerStdio
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from openai import AsyncOpenAI
import os

In [ ]:
load_dotenv(override=True)

In [ ]:
params = {"command": "uv", "args": ["run", "math_server.py"]}
#params = {"command": "python", "args": ["math_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=20) as client:
    tools = await client.list_tools()
    resources = await client.session.list_resources()

print(tools)
print()
print(resources)

In [ ]:
instructions = """
You are a mathematician who can solve arithmetic problems intelligently and also tell math jokes. 
when asked to tell a joke, use resources from mcp server.
"""

client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=client)

async with MCPServerStdio(params=params, client_session_timeout_seconds=20) as server:
    agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model=model,
            mcp_servers=[server]
            )
    
    with trace("Mathematician"):
            result = await Runner.run(agent, "what is 2 + 3 + 7")
            print(result.final_output)